In [31]:
pip install yfinance pandas numpy requests matplotlib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


Load Phase 1 data

In [32]:
# index_col tells pandas to use the first column as the index of the DataFrame, which is useful for time series data where the first column often contains dates. 
# parse_dates tells pandas to convert the index column to datetime objects instead of leaving them as strings, which allows for easier manipulation and analysis of time series data.

import pandas as pd
import numpy as np
import yfinance as yf

prices = pd.read_csv(
    "../data/raw/prices.csv",
    index_col=0,
    parse_dates=True
)

market = pd.read_csv(
    "../data/raw/market.csv",
    index_col=0,
    parse_dates=True
)

print(prices.head())
print(market.head())

                  AAPL        AMZN       GOOGL         JPM        META  \
Date                                                                     
2025-01-02  242.301926  220.220001  188.445709  231.734146  596.292725   
2025-01-03  241.815033  224.190002  190.793427  234.901169  601.656128   
2025-01-06  243.444626  227.610001  195.847061  233.755951  627.100464   
2025-01-07  240.672333  222.110001  194.474228  236.007614  614.851013   
2025-01-08  241.159210  222.130005  192.942215  235.968781  607.716187   

                  MSFT        NVDA        TSLA         UNH         XOM  
Date                                                                    
2025-01-02  413.672638  138.103714  379.279999  485.811096  102.132416  
2025-01-03  418.386688  144.254547  410.440002  493.986450  102.655884  
2025-01-06  422.833984  149.207153  411.049988  494.631561  102.541679  
2025-01-07  417.418182  139.930984  394.359985  495.190125  103.502937  
2025-01-08  419.582520  139.901062  394.940

Adjust prices for splits and dividends

In [33]:
testing_phase_1 = yf.download(
    "AAPL",
    start="2020-01-01",
    end="2025-01-01",
    auto_adjust=True
)

print(testing_phase_1.head())

[*********************100%***********************]  1 of 1 completed

Price           Close       High        Low       Open     Volume
Ticker           AAPL       AAPL       AAPL       AAPL       AAPL
Date                                                             
2020-01-02  72.333855  72.394063  71.091161  71.344032  135480400
2020-01-03  71.630638  72.389257  71.406666  71.563205  146322800
2020-01-06  72.201416  72.239950  70.503554  70.754021  118387200
2020-01-07  71.861847  72.466330  71.642689  72.211049  108872000
2020-01-08  73.017838  73.318877  71.565621  71.565621  132079200


Check for missing values

In [34]:
print(prices.isna().sum())
print(prices.isna().sum().sum())

AAPL     0
AMZN     0
GOOGL    0
JPM      0
META     0
MSFT     0
NVDA     0
TSLA     0
UNH      0
XOM      0
dtype: int64
0


Align trading dates

In [35]:
prices = prices.dropna()
print(prices.isna().sum())

AAPL     0
AMZN     0
GOOGL    0
JPM      0
META     0
MSFT     0
NVDA     0
TSLA     0
UNH      0
XOM      0
dtype: int64


if there are only a few missing prices

In [36]:
# ffill() is a method in pandas that fills missing values in a DataFrame or Series by propagating the last valid observation forward. This means that if there are any NaN (Not a Number) values in the data, they will be replaced with the most recent non-NaN value that precedes them. This is particularly useful in time series data where you want to maintain continuity and avoid gaps caused by missing data points.

prices = prices.ffill()
prices = prices.dropna()

Remove stale prices

In [37]:
unchanged = prices.pct_change() == 0

for stock in prices.columns:
    zero_returns = (prices[stock].pct_change() == 0).sum()
    print(stock, zero_returns)

AAPL 0
AMZN 0
GOOGL 0
JPM 1
META 0
MSFT 0
NVDA 0
TSLA 0
UNH 0
XOM 1


Convert prices to log returns

In [38]:
log_returns = np.log(prices / prices.shift(1))
log_returns = log_returns.dropna()

print(log_returns.head())

                AAPL      AMZN     GOOGL       JPM      META      MSFT  \
Date                                                                     
2025-01-03 -0.002011  0.017867  0.012381  0.013574  0.008954  0.011331   
2025-01-06  0.006716  0.015140  0.026143 -0.004887  0.041421  0.010574   
2025-01-07 -0.011453 -0.024461 -0.007034  0.009586 -0.019727 -0.012891   
2025-01-08  0.002021  0.000090 -0.007909 -0.000165 -0.011672  0.005172   
2025-01-10 -0.024399 -0.014465 -0.009897 -0.013499  0.008381 -0.013302   

                NVDA      TSLA       UNH       XOM  
Date                                                
2025-01-03  0.043574  0.078955  0.016688  0.005112  
2025-01-06  0.033756  0.001485  0.001305 -0.001113  
2025-01-07 -0.064186 -0.041451  0.001129  0.009331  
2025-01-08 -0.000214  0.001470  0.019774 -0.016877  
2025-01-10 -0.030435 -0.000507 -0.007329 -0.003654  


Winsorize extreme returns

In [39]:
lower = log_returns.quantile(0.01)
upper = log_returns.quantile(0.99)

clean_returns = log_returns.clip(
    lower=lower,
    upper=upper,
    axis=1
)

In [40]:
USE_WINSORIZATION = False

In [41]:
if USE_WINSORIZATION:

    lower = log_returns.quantile(0.01)
    upper = log_returns.quantile(0.99)

    clean_returns = log_returns.clip(
        lower=lower,
        upper=upper,
        axis=1
    )

else:
    clean_returns = log_returns.copy()

Create portfolio weights

In [42]:
weights = {
    "AAPL": 0.10,
    "MSFT": 0.10,
    "NVDA": 0.10,
    "JPM": 0.10,
    "XOM": 0.10,
    "AMZN": 0.10,
    "META": 0.10,
    "GOOGL": 0.10,
    "TSLA": 0.10,
    "UNH": 0.10
}

In [43]:
print(sum(weights.values()))

1.0


Build weighted portfolio returns

In [44]:
# .mul() is a method in pandas that performs element-wise multiplication between two DataFrames or a DataFrame and a Series. In this case, it multiplies the clean_returns DataFrame by the weights_series, aligning them by their index (dates) and columns (stock tickers). The axis=1 argument specifies that the multiplication should be done along the columns (i.e., for each stock).
# .sum(axis=1) is a method that sums the values along the specified axis. In this case, axis=1 means that it sums across the columns (i.e., for each date), resulting in a single Series of portfolio returns for each date.


weights_series = pd.Series(weights)

portfolio_returns = clean_returns.mul(
    weights_series,
    axis=1
).sum(axis=1)

Calculate rolling volatility

In [45]:
rolling_volatility = portfolio_returns.rolling(
    window=30
).std()

In [46]:
rolling_volatility = (
    portfolio_returns
    .rolling(window=30)
    .std()
    * np.sqrt(252)
)

 DISPLAY RESULTS

In [47]:
print("\nClean Prices:")
print(prices.head())

print("\nClean Returns:")
print(clean_returns.head())

print("\nPortfolio Returns:")
print(portfolio_returns.head())

print("\nRolling Volatility:")
print(rolling_volatility.head(35))


Clean Prices:
                  AAPL        AMZN       GOOGL         JPM        META  \
Date                                                                     
2025-01-02  242.301926  220.220001  188.445709  231.734146  596.292725   
2025-01-03  241.815033  224.190002  190.793427  234.901169  601.656128   
2025-01-06  243.444626  227.610001  195.847061  233.755951  627.100464   
2025-01-07  240.672333  222.110001  194.474228  236.007614  614.851013   
2025-01-08  241.159210  222.130005  192.942215  235.968781  607.716187   

                  MSFT        NVDA        TSLA         UNH         XOM  
Date                                                                    
2025-01-02  413.672638  138.103714  379.279999  485.811096  102.132416  
2025-01-03  418.386688  144.254547  410.440002  493.986450  102.655884  
2025-01-06  422.833984  149.207153  411.049988  494.631561  102.541679  
2025-01-07  417.418182  139.930984  394.359985  495.190125  103.502937  
2025-01-08  419.582520  139.

In [48]:
prices.to_csv("../data/clean_prices.csv")

In [49]:
clean_returns.to_csv("../data/clean_returns.csv")

In [50]:
portfolio_returns.to_csv(
    "../data/portfolio_returns.csv",
    header=True
)

In [51]:
rolling_volatility.to_csv(
    "../data/rolling_volatility.csv",
    header=True
)

validation checks

In [52]:
print("========== VALIDATION ==========")

print("Price shape:", prices.shape)

print("Return shape:", clean_returns.shape)

print(
    "Missing prices:",
    prices.isna().sum().sum()
)

print(
    "Missing returns:",
    clean_returns.isna().sum().sum()
)

print(
    "Portfolio return observations:",
    len(portfolio_returns)
)

print(
    "Weight sum:",
    # weights.sum()
)

print(
    "Minimum price:",
    prices.min().min()
)

print(
    "Maximum price:",
    prices.max().max()
)

========== VALIDATION ==========
Price shape: (249, 10)
Return shape: (248, 10)
Missing prices: 0
Missing returns: 0
Portfolio return observations: 248
Weight sum:
Minimum price: 94.17801666259766
Maximum price: 787.419189453125
